Task 6 - Tableau Dashboards and Insights

In [1]:
#Import required libraries for pyspark
from pyspark.sql import SparkSession
from pyspark.ml import PipelineModel
from pyspark.sql.functions import col, when

In [2]:
# Initialising spark session
spark = SparkSession.builder \
    .appName("FraudDetectionTask3") \
    .master("local[2]") \
    .config("spark.driver.memory", "6g") \
    .getOrCreate()
print("Spark session created successfully")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/29 11:17:24 WARN Utils: Your hostname, Sais-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.223.134.171 instead (on interface en0)
26/06/29 11:17:24 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/29 11:17:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/29 11:17:25 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark session created successfully


In [3]:
# Load processed data
processed_df = spark.read.parquet("task2_processed_data")

In [4]:
# Load pipeline model
pipeline_model = PipelineModel.load("task2_pipeline")

In [5]:
# Check whether processed data is loaded correctly
processed_df.printSchema()
processed_df.show(5)

root
 |-- label: double (nullable = true)
 |-- c1: double (nullable = true)
 |-- c2: double (nullable = true)
 |-- c3: double (nullable = true)
 |-- c4: double (nullable = true)
 |-- c5: double (nullable = true)
 |-- c6: double (nullable = true)
 |-- c7: double (nullable = true)
 |-- c8: double (nullable = true)
 |-- c9: double (nullable = true)
 |-- c10: double (nullable = true)
 |-- c11: double (nullable = true)
 |-- c12: double (nullable = true)
 |-- c13: double (nullable = true)
 |-- c14: double (nullable = true)
 |-- c15: double (nullable = true)
 |-- c16: double (nullable = true)
 |-- c17: double (nullable = true)
 |-- c18: double (nullable = true)
 |-- c19: double (nullable = true)
 |-- c20: double (nullable = true)
 |-- c21: double (nullable = true)
 |-- c22: double (nullable = true)
 |-- c23: double (nullable = true)
 |-- c24: double (nullable = true)
 |-- c25: double (nullable = true)
 |-- c26: double (nullable = true)
 |-- c27: double (nullable = true)
 |-- c28: double (null

26/06/29 11:17:30 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----+------------------+-------------------+-------------------+------------------+-------------------+------------------+--------------------+-------------------+------------------+------------------+-------------------+-------------------+------------------+------------------+--------------------+-------------------+-----------------+-------------------+--------------------+--------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+--------------------+--------------------+
|label|                c1|                 c2|                 c3|                c4|                 c5|                c6|                  c7|                 c8|                c9|               c10|                c11|                c12|               c13|               c14|                 c15|                c16|              c17|                c18|                 c19|                 c20|  

In [6]:
# Split dataset into 70% training and 30% testing
train_data, test_data = processed_df.randomSplit([0.7, 0.3], seed=42)

In [7]:
# Import required libraries
from pyspark.ml.classification import (
    LogisticRegressionModel,
    DecisionTreeClassificationModel,
    RandomForestClassificationModel,
    GBTClassificationModel
)

# Load the saved models to a new variable
lr_model = LogisticRegressionModel.load("lr_model")
dt_model = DecisionTreeClassificationModel.load("dt_model")
rf_model = RandomForestClassificationModel.load("rf_model")
gbt_model = GBTClassificationModel.load("gbt_model")

print("Saved models loaded successfully")

Saved models loaded successfully


In [8]:
# Generate Predictions
lr_predictions = lr_model.transform(test_data)
dt_predictions = dt_model.transform(test_data)
rf_predictions = rf_model.transform(test_data)
gbt_predictions = gbt_model.transform(test_data)

print("Predictions generated")

Predictions generated


Prepare Data for Tabeleau

In [9]:
# Select only required columns for Tableau
lr_tableau = lr_predictions.select("label", "prediction").toPandas()
dt_tableau = dt_predictions.select("label", "prediction").toPandas()
rf_tableau = rf_predictions.select("label", "prediction").toPandas()
gbt_tableau = gbt_predictions.select("label", "prediction").toPandas()

print("Data prepared for Tableau")

26/06/29 11:17:51 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/06/29 11:18:38 WARN DAGScheduler: Broadcasting large task binary with size 1171.1 KiB
                                                                                

Data prepared for Tableau


Export data as csv files

In [ ]:
# Export data as csv files for Tableau dashboards
lr_tableau.to_csv("lr_predictions.csv", index=False)
dt_tableau.to_csv("dt_predictions.csv", index=False)
rf_tableau.to_csv("rf_predictions.csv", index=False)
gbt_tableau.to_csv("gbt_predictions.csv", index=False)

print("CSV files exported successfully")

In [10]:
# Recreate evaluators

from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)

# AUC - measures how well the model separates classes
auc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    metricName="areaUnderROC"
)

# Accuracy - overall correctness of the prediction
accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

# F1 Score - shows how well the model balances correct predictions and missed cases
f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

# Precision – shows how many of the predicted positive cases are actually correct
precision_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

In [11]:
# Compute the Metrics Evaluators

# Logistic Regression Model
lr_accuracy = accuracy_evaluator.evaluate(lr_predictions)
lr_auc = auc_evaluator.evaluate(lr_predictions)
lr_f1 = f1_evaluator.evaluate(lr_predictions)
lr_precision = precision_evaluator.evaluate(lr_predictions)

# Decision Tree Model
dt_accuracy = accuracy_evaluator.evaluate(dt_predictions)
dt_auc = auc_evaluator.evaluate(dt_predictions)
dt_f1 = f1_evaluator.evaluate(dt_predictions)
dt_precision = precision_evaluator.evaluate(dt_predictions)

# Random Forest Model
rf_accuracy = accuracy_evaluator.evaluate(rf_predictions)
rf_auc = auc_evaluator.evaluate(rf_predictions)
rf_f1 = f1_evaluator.evaluate(rf_predictions)
rf_precision = precision_evaluator.evaluate(rf_predictions)

# GBT Model
gbt_accuracy = accuracy_evaluator.evaluate(gbt_predictions)
gbt_auc = auc_evaluator.evaluate(gbt_predictions)
gbt_f1 = f1_evaluator.evaluate(gbt_predictions)
gbt_precision = precision_evaluator.evaluate(gbt_predictions)

26/06/29 11:21:52 WARN DAGScheduler: Broadcasting large task binary with size 1191.2 KiB
26/06/29 11:22:13 WARN DAGScheduler: Broadcasting large task binary with size 1178.2 KiB
26/06/29 11:22:39 WARN DAGScheduler: Broadcasting large task binary with size 1191.2 KiB
26/06/29 11:22:56 WARN DAGScheduler: Broadcasting large task binary with size 1191.2 KiB
                                                                                

In [12]:
# Full metrics comparission table of all the 4 models

import pandas as pd

# Create a list containing evaluation metrics results of all 4 ML models
data = [
    ("Logistic Regression", lr_accuracy, lr_auc, lr_f1, lr_precision),
    ("Decision Tree", dt_accuracy, dt_auc, dt_f1, dt_precision),
    ("Random Forest", rf_accuracy, rf_auc, rf_f1, rf_precision),
    ("GBT Classifier", gbt_accuracy, gbt_auc, gbt_f1, gbt_precision)
]

# Define column names for the summary table
columns = ["Model", "Accuracy", "AUC", "F1 Score", "Precision"]

# Convert the data into a pandas DataFrame and display the table
summary_df = pd.DataFrame(data, columns=columns)
summary_df

,Model,Accuracy,AUC,F1 Score,Precision
0,Logistic Regression,0.635175,0.679043,0.629832,0.635862
1,Decision Tree,0.704703,0.669855,0.704703,0.704703
2,Random Forest,0.697418,0.769555,0.696473,0.697084
3,GBT Classifier,0.718196,0.795335,0.718085,0.718026


Export Model Metrics

In [14]:
# Export model performance table
summary_df.to_csv("model_metrics.csv", index=False)

print("Metrics exported for Tableau")

Metrics exported for Tableau


Extract Feature Importance

In [17]:
# Feature selection
feature_cols = [f"c{i}" for i in range(1, 29)]

In [18]:
import pandas as pd

# Feature names in same order used in VectorAssembler
feature_names = feature_cols

# Random Forest Feature Importance
rf_importance = rf_model.featureImportances.toArray()

rf_df = pd.DataFrame({
    "Feature": feature_names,
    "Random Forest": rf_importance
})

# GBT Feature Importance
gbt_importance = gbt_model.featureImportances.toArray()

gbt_df = pd.DataFrame({
    "Feature": feature_names,
    "GBT": gbt_importance
})

# Decision Tree Feature Importance
dt_importance = dt_model.featureImportances.toArray()

dt_df = pd.DataFrame({
    "Feature": feature_names,
    "Decision Tree": dt_importance
})

# Logistic Regression (coefficients = importance proxy)
lr_importance = lr_model.coefficients.toArray()

lr_df = pd.DataFrame({
    "Feature": feature_names,
    "Logistic Regression": abs(lr_importance)
})

# Merge all models into one table
final_fi = rf_df.merge(gbt_df, on="Feature") \
                .merge(dt_df, on="Feature") \
                .merge(lr_df, on="Feature")

# Sort by RF importance
final_fi = final_fi.sort_values(by="Random Forest", ascending=False)

# Export for Tableau
final_fi.to_csv("all_feature_importance.csv", index=False)

print("All feature importance exported successfully")

All feature importance exported successfully


Singal vs Noise Distribution

In [8]:
# Export HIGGS dataset class distribution to csv file

from pyspark.sql.functions import col

# Create class distribution (Signal vs Noise)
class_dist = processed_df.groupBy("label").count()

# Rename columns for Tableau clarity
class_dist = class_dist.withColumnRenamed("label", "Class (0=Noise, 1=Signal)") \
                       .withColumnRenamed("count", "Count")

# Save for Tableau
class_dist.toPandas().to_csv("higgs_signal_noise_distribution.csv", index=False)

print("Signal vs Noise distribution exported successfully")

[Stage 10:===========================================>            (28 + 2) / 36]

Signal vs Noise distribution exported successfully


Extract Model Confidence for all Models

In [20]:
# Helper function to extract confidence 
def get_confidence(df):
    return df.select("probability").toPandas()["probability"].apply(lambda x: float(max(x)))

# Extract confidence for each model
lr_conf = get_confidence(lr_predictions)
dt_conf = get_confidence(dt_predictions)
rf_conf = get_confidence(rf_predictions)
gbt_conf = get_confidence(gbt_predictions)

# Combine into one DataFrame for Tableau
confidence_df = pd.DataFrame({
    "Logistic Regression": lr_conf,
    "Decision Tree": dt_conf,
    "Random Forest": rf_conf,
    "GBT": gbt_conf
})

# Export for Tableau
confidence_df.to_csv("model_confidence_all.csv", index=False)

print("Model confidence exported successfully")

26/06/28 08:44:04 WARN DAGScheduler: Broadcasting large task binary with size 1174.7 KiB
                                                                                

Model confidence exported successfully


In [14]:
# Signal vs Noise Feature Shift

signal_noise_shift = processed_df.groupBy("label").mean()

signal_noise_shift.toPandas().to_csv("signal_noise_shift.csv", index=False)

print("Signal vs Noise feature shift exported")

[Stage 100:=============================================>         (30 + 2) / 36]

Signal vs Noise feature shift exported


Data Quality and Pipeline Monitoring

In [17]:
from pyspark.sql.functions import col, sum, when
import pandas as pd

# Missing values
missing = processed_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in processed_df.columns
]).toPandas().T.reset_index()

missing.columns = ["Feature", "Missing_Values"]

# Data types
types = pd.DataFrame({
    "Feature": processed_df.columns,
    "Data_Type": [str(f.dataType) for f in processed_df.schema.fields]
})

# Summary statistics
summary = processed_df.describe().toPandas()

# Dataset overview
overview = pd.DataFrame({
    "Metric": ["Total Records", "Total Features"],
    "Value": [processed_df.count(), len(processed_df.columns) - 1]
})

# Write everything into one Excel workbook 
with pd.ExcelWriter("data_quality_pipeline_monitoring.xlsx") as writer:
    missing.to_excel(writer, sheet_name="Missing Values", index=False)
    types.to_excel(writer, sheet_name="Data Types", index=False)
    summary.to_excel(writer, sheet_name="Feature Summary", index=False)
    overview.to_excel(writer, sheet_name="Dataset Overview", index=False)

print("Data Quality & Pipeline Monitoring workbook exported successfully.")

Data Quality & Pipeline Monitoring workbook exported successfully.


Scalibility and Cost

In [18]:
import pandas as pd

scale_df = pd.DataFrame({
    "Metric": [
        "Total Rows",
        "Total Features",
        "Approx Memory (MB)"
    ],
    "Value": [
        processed_df.count(),
        len(processed_df.columns) - 1,
        processed_df.count() * len(processed_df.columns) * 8 / (1024 * 1024)  # rough estimate
    ]
})

scale_df.to_csv("scalability_metrics.csv", index=False)

print("Scalability metrics exported")

Scalability metrics exported


In [19]:
from pyspark.sql.functions import countDistinct

feature_complexity = processed_df.select([
    countDistinct(c).alias(c)
    for c in processed_df.columns if c != "label"
])

feature_complexity.toPandas().to_csv("feature_complexity.csv", index=False)

print("Feature complexity exported")

[Stage 121:====================================================> (98 + 2) / 100]

Feature complexity exported


In [21]:
label_dist = processed_df.groupBy("label").count()

label_dist = label_dist.withColumnRenamed("label", "Class (0=Noise, 1=Signal)") \
                       .withColumnRenamed("count", "Count")

label_dist.toPandas().to_csv("labelDistribution.csv", index=False)

print("Label distribution exported")

Label distribution exported
